In [8]:
import pandas as pd
pd.options.plotting.backend = "plotly"
from summer2 import CompartmentalModel
from summer2.parameters import Parameter, DerivedOutput

In [9]:
# Declare a function that takes a dictionary with configurations 
# and returns a summer model

# This is the function described in 02_basic_model_intro.ipynb in the textbook:
def build_sir_model(
    config: dict,
) -> CompartmentalModel:
    """ 
    Args: 
        config: Quantities other than parameters that we need for model construction
    Returns:
        model: Summer model object
    """

    # First we define the compartments that this model has
    compartments = (
        "susceptible",
        "infectious",
        "recovered",
    )
    # We want to be able to define how many time units the model runs over
    analysis_times = (0.0, config["end_time"]) 

    # Summer syntax to use the CompartmentalModel object for our model
    model = CompartmentalModel(
        times=analysis_times,
        compartments=compartments,
        infectious_compartments=("infectious",),
    )

    # We define how the population is distributed at time 0
    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"],
        },
    )

    # We define the directions and names of flows between compartments
    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    return model


In [10]:
# Now we declare the model configurations as a dictionary:
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau
    "seed": 144.0, # Number of cases diagnosed in 2004
    "end_time": 17.0, # Say time unit is years, and we end by December, 2020 
}

# Build simple SIR model with these configs:
sir_model = build_sir_model(model_config)

In [11]:
# So we built the model, but we need to specify the parameters, ie. the disease dynamics:
parameters = {
    "recovery": 0.15, # -Ish for untreated TB per year
    "contact_rate": 10, # Effective contacts per person per year
}

# Run model with the specified parameters
sir_model.run(parameters=parameters)
compartment_values = sir_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

Not surprisingly, the simple SIR model does not capture realistic TB dynamics.